# Brain Network Analysis — Balanced Performance (v5-light)
**Upgrades over v4:**
- LightGBM + XGBoost with tuned hyperparameters
- BorderlineSMOTE for imbalanced tasks
- Spectral features (Laplacian eigenvalues)
- Hemispheric asymmetry features
- Optimal threshold tuning (Youden's J)
- All results printed, no CSV files


In [1]:

"""
Brain Network Analysis — Maximum Performance Pipeline (v5)
"""
from __future__ import annotations
import warnings, gc, itertools
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy import sparse
from scipy.stats import ttest_ind

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score, average_precision_score, roc_curve)
from sklearn.impute import SimpleImputer

try:
    from xgboost import XGBClassifier; HAS_XGB = True
except ImportError:
    HAS_XGB = False; print("[WARN] XGBoost not available")

try:
    from lightgbm import LGBMClassifier; HAS_LGB = True
except ImportError:
    HAS_LGB = False; print("[WARN] LightGBM not available")

try:
    from imblearn.over_sampling import BorderlineSMOTE, SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False; ImbPipeline = Pipeline
    print("[WARN] imbalanced-learn not available")

from statsmodels.stats.multitest import fdrcorrection
import networkx as nx

warnings.filterwarnings("ignore")

try:
    import torch, torch.nn as nn, torch.nn.functional as F; HAS_TORCH = True
except Exception:
    HAS_TORCH = False; torch = None
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader as PyGDataLoader
    from torch_geometric.nn import GCNConv, GINConv, global_mean_pool, BatchNorm
    HAS_PYG = True
except Exception:
    HAS_PYG = False


In [2]:

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
MAT_DIR   = r"C:\Users\vishn\Downloads\brainnetworks\smallgraphs"
META_PATH = r"C:\Users\vishn\Downloads\brainnetworks\metainfo.xls"
OUTPUT_DIR = Path(r"C:\Users\vishn\Downloads\brainnetworks\brain_network_outputs_v5")

SEED            = 42
N_SPLITS_OUTER  = 5
N_SPLITS_INNER  = 3
N_NODES         = 70
GNN_EPOCHS      = 200
GNN_PATIENCE    = 25
GNN_LR          = 1e-3
GNN_WEIGHT_DECAY= 1e-4
GNN_HIDDEN      = 64
GNN_BATCH_SIZE  = 16

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if (HAS_TORCH and torch.cuda.is_available()) else "cpu") if HAS_TORCH else None
print(f"Device: {DEVICE}")
print(f"LightGBM={HAS_LGB}  XGBoost={HAS_XGB}  imblearn={HAS_IMBLEARN}")


Device: cuda
LightGBM=True  XGBoost=True  imblearn=True


In [3]:

# ─── GRAPH LOADING & NORMALIZATION ────────────────────────────────────────────
def ensure_square(a, n=N_NODES):
    if a.shape != (n, n):
        raise ValueError(f"Expected {n}x{n}, got {a.shape}")
    return a

def load_fibergraph(path: Path) -> np.ndarray:
    d = loadmat(path)
    A = d["fibergraph"]
    if sparse.issparse(A): A = A.toarray()
    A = np.asarray(A, dtype=np.float64)
    ensure_square(A)
    A = 0.5 * (A + A.T)
    np.fill_diagonal(A, 0.0)
    A[A < 0] = 0.0
    return A

def normalize_row_prob(A: np.ndarray) -> np.ndarray:
    rs = A.sum(axis=1, keepdims=True); rs[rs == 0] = 1.0
    W = A / rs; np.fill_diagonal(W, 0.0); return W

def upper_triangle_features(Ws: np.ndarray) -> np.ndarray:
    idx = np.triu_indices(Ws.shape[1], k=1)
    return Ws[:, idx[0], idx[1]]


In [4]:

# ─── FEATURE ENGINEERING ──────────────────────────────────────────────────────
def build_graph(W: np.ndarray, q: float = 0.25) -> nx.Graph:
    nz = W[W > 0]
    thr = np.quantile(nz, q) if nz.size else 0.0
    A = W.copy(); A[A < thr] = 0.0
    G = nx.from_numpy_array(A)
    for _, _, d in G.edges(data=True):
        d["distance"] = 1.0 / (float(d["weight"]) + 1e-8)
    return G

def _stat(arr, prefix):
    a = np.array(arr, dtype=float)
    return {f"{prefix}_mean": np.nanmean(a), f"{prefix}_std": np.nanstd(a),
            f"{prefix}_min": np.nanmin(a),   f"{prefix}_max": np.nanmax(a),
            f"{prefix}_p25": np.nanpercentile(a, 25),
            f"{prefix}_p75": np.nanpercentile(a, 75)}

def compute_spectral_features(W: np.ndarray) -> Dict[str, float]:
    try:
        G = build_graph(W)
        L = nx.normalized_laplacian_matrix(G, weight="weight").toarray()
        evals = np.sort(np.real(np.linalg.eigvalsh(L)))
        feats = {
            "spec_algebraic_conn": float(evals[1]) if len(evals) > 1 else 0.0,
            "spec_spectral_gap":   float(evals[1] - evals[0]) if len(evals) > 1 else 0.0,
            "spec_largest_eval":   float(evals[-1]),
            "spec_energy":         float(np.sum(evals**2)),
            "spec_entropy":        float(-np.sum(np.where(evals>0, evals*np.log(evals+1e-12), 0))),
        }
        for i, v in enumerate(evals[1:11]):
            feats[f"spec_eval_{i+1}"] = float(v)
        feats.update(_stat(evals, "spec_all"))
        return feats
    except Exception:
        return {}

def compute_graph_features(W: np.ndarray) -> Dict[str, float]:
    G = build_graph(W)
    degrees   = np.array([d for _, d in G.degree(weight=None)], float)
    strengths = np.array([d for _, d in G.degree(weight="weight")], float)
    try: cc = list(nx.clustering(G, weight="weight").values())
    except: cc = [0.0]*N_NODES
    try: bw = list(nx.betweenness_centrality(G, weight="distance").values())
    except: bw = [0.0]*N_NODES
    try: cl = list(nx.closeness_centrality(G, distance="distance").values())
    except: cl = [0.0]*N_NODES
    try: ev = list(nx.eigenvector_centrality_numpy(G, weight="weight").values())
    except: ev = [0.0]*N_NODES
    try:
        comms = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
        modularity = nx.algorithms.community.quality.modularity(G, comms, weight="weight")
    except: modularity = np.nan
    try:
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        aspl = nx.average_shortest_path_length(G.subgraph(comps[0]), weight="distance")
    except: aspl = np.nan
    feats = {
        "density": nx.density(G), "modularity": modularity,
        "n_components": nx.number_connected_components(G),
        "aspl": aspl, "global_eff": nx.global_efficiency(G),
        "transitivity": nx.transitivity(G),
        **_stat(degrees, "degree"), **_stat(strengths, "strength"),
        **_stat(cc, "clustering"), **_stat(bw, "betweenness"),
        **_stat(cl, "closeness"),  **_stat(ev, "eigenvec"),
    }
    feats.update(compute_spectral_features(W))
    return feats

def compute_asymmetry_features(W: np.ndarray) -> np.ndarray:
    """
    Inter-hemispheric asymmetry: difference between upper-left and upper-right
    quadrant connectivity (assuming first 35 nodes = left, last 35 = right).
    """
    half = N_NODES // 2
    L_block = W[:half, :half]   # left-left
    R_block = W[half:, half:]   # right-right
    LR_block = W[:half, half:]  # left-right (interhemispheric)

    def quad_stats(Q):
        vals = Q[Q > 0]
        if vals.size == 0: return np.zeros(5)
        return np.array([vals.mean(), vals.std(), np.percentile(vals,25),
                         np.percentile(vals,75), np.count_nonzero(Q)/Q.size])

    diff = quad_stats(L_block) - quad_stats(R_block)
    inter = quad_stats(LR_block)
    return np.concatenate([diff, inter])   # 10 features


In [5]:

# ─── DATASET LOADING ──────────────────────────────────────────────────────────
@dataclass
class DatasetBundle:
    df:          pd.DataFrame
    W_prob:      np.ndarray    # (N, 70, 70)
    X_edges:     np.ndarray    # 2415 upper-triangle weights
    X_graph:     pd.DataFrame  # graph + spectral summary stats
    X_spectral:  np.ndarray    # spectral only
    X_asym:      np.ndarray    # hemispheric asymmetry (10)
    X_all:       np.ndarray    # everything combined

def load_dataset(mat_dir: str, meta_path: str) -> DatasetBundle:
    mat_dir = Path(mat_dir); meta_path = Path(meta_path)
    df = pd.read_excel(meta_path)
    df.columns = [str(c).strip() for c in df.columns]
    mat_files = sorted(mat_dir.glob("*.mat"))
    file_map  = {f.stem.replace("_fiber",""):f.name for f in mat_files}
    df["URSI"] = df["URSI"].astype(str).str.strip()
    df = df[df["URSI"].isin(file_map)].drop_duplicates("URSI").copy()
    df["filename"] = df["URSI"].map(file_map)
    df = df.reset_index(drop=True)

    W_list, graph_list, spec_list, asym_list = [], [], [], []
    for _, row in df.iterrows():
        A = load_fibergraph(mat_dir / row["filename"])
        W = normalize_row_prob(A)
        W_list.append(W)
        graph_list.append(compute_graph_features(W))
        spec_list.append(compute_spectral_features(W))
        asym_list.append(compute_asymmetry_features(W))

    W_prob    = np.stack(W_list)
    X_edges   = upper_triangle_features(W_prob)
    X_graph   = pd.DataFrame(graph_list).fillna(0)
    X_spectral= pd.DataFrame(spec_list).fillna(0).values
    X_asym    = np.nan_to_num(np.stack(asym_list))
    X_all     = np.hstack([X_edges, X_graph.values, X_asym])

    print(f"Loaded {len(df)} subjects")
    print(f"  Edge features:      {X_edges.shape[1]}")
    print(f"  Graph+Spec stats:   {X_graph.shape[1]}")
    print(f"  Asymmetry:          {X_asym.shape[1]}")
    print(f"  TOTAL combined:     {X_all.shape[1]}")
    return DatasetBundle(df, W_prob, X_edges, X_graph, X_spectral, X_asym, X_all)


In [6]:

# ─── TASK ENCODING ────────────────────────────────────────────────────────────
@dataclass
class TaskSpec:
    name:          str
    mask:          np.ndarray
    y:             np.ndarray
    class_names:   List[str]
    is_imbalanced: bool = False

def encode_sex(s: pd.Series):
    m = {"m":1,"male":1,"1":1,"man":1,"f":0,"female":0,"0":0,"woman":0}
    return np.array([m[x.strip().lower()] for x in s.astype(str)], int)

def build_tasks(df: pd.DataFrame) -> List[TaskSpec]:
    y_sex = encode_sex(df["Sex"])
    tasks = [TaskSpec("sex_female_vs_male", np.ones(len(df),bool), y_sex,
                      ["Female","Male"], is_imbalanced=False)]
    st = pd.to_numeric(df["Subject_type"], errors="coerce")
    cr_mask = st.isin([0,1]).values; cr_y = st.loc[cr_mask].astype(int).values
    mh_mask = st.isin([0,2]).values; mh_y = (st.loc[mh_mask].astype(int).values==2).astype(int)
    tasks.append(TaskSpec("creativity_normal_vs_creative", cr_mask, cr_y,
                          ["Normal","Creative"], is_imbalanced=True))
    tasks.append(TaskSpec("math_normal_vs_high", mh_mask, mh_y,
                          ["Normal","High Math"], is_imbalanced=True))
    for t in tasks:
        pos=t.y.sum(); neg=len(t.y)-pos
        print(f"  {t.name}: n={len(t.y)}, class0={neg}, class1={pos}")
    return tasks


In [7]:

# ─── HELPERS ──────────────────────────────────────────────────────────────────
def _cap(X_train):
    Xt = SimpleImputer(strategy="median").fit_transform(X_train)
    try: Xt = VarianceThreshold(1e-6).fit_transform(Xt)
    except: pass
    return Xt.shape[0], (Xt.shape[1] if Xt.ndim==2 else 0)

def adapt_grid(name, grid, X_train):
    ns, nf = _cap(X_train)
    if nf < 1: return None
    g = {k: list(v) for k,v in grid.items()}
    for k, vs in list(g.items()):
        if k.endswith("__k"):
            g[k] = [v for v in vs if 1<=int(v)<=nf] or [min(nf, max(vs))]
        elif k == "pca__n_components" or k == "svd__n_components":
            mx = min(ns-1, nf)
            g[k] = [v for v in vs if 1<=int(v)<=mx] or [max(1,mx)]
        elif k == "select__max_features":
            g[k] = [v for v in vs if 1<=int(v)<=nf] or [min(nf,50)]
    return g if all(g.values()) else None

def evaluate_metrics(y_true, y_pred, y_score):
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "f1": f1_score(y_true, y_pred, zero_division=0)}
    try: m["roc_auc"] = roc_auc_score(y_true, y_score)
    except: m["roc_auc"] = np.nan
    try: m["avg_precision"] = average_precision_score(y_true, y_score)
    except: m["avg_precision"] = np.nan
    return m

def find_best_threshold(y_true, y_score):
    fpr, tpr, thr = roc_curve(y_true, y_score)
    return float(thr[np.argmax(tpr - fpr)])


In [8]:

# ─── MODEL SPACES ─────────────────────────────────────────────────────────────
def get_model_spaces(is_imbalanced: bool = False, pos_ratio: float = 0.23):
    PipeClass = ImbPipeline if HAS_IMBLEARN else Pipeline
    # BorderlineSMOTE is more targeted than vanilla SMOTE
    smote_k = 1
    if HAS_IMBLEARN and is_imbalanced:
        try:    smote_steps = [("smote", BorderlineSMOTE(random_state=SEED, k_neighbors=smote_k))]
        except: smote_steps = [("smote", SMOTE(random_state=SEED, k_neighbors=smote_k))]
    else:
        smote_steps = []

    spw = (1 - pos_ratio) / pos_ratio   # scale_pos_weight for boosters
    cw  = "balanced"
    spaces = {}

    # ── SVM (RBF) + MI selection ──────────────────────────────────────────────
    spaces["svm_mi"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                   *smote_steps,
                   ("sel", SelectKBest(mutual_info_classif)),
                   ("clf", SVC(kernel="rbf", probability=True, class_weight=cw, random_state=SEED))]),
        {"sel__k": [50,100,200], "clf__C": [0.1,1,10,100],
         "clf__gamma": ["scale",0.01,0.1]},
    )

    # ── SVM + PCA ─────────────────────────────────────────────────────────────
    spaces["svm_pca"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                   *smote_steps,
                   ("pca", PCA(random_state=SEED)),
                   ("clf", SVC(kernel="rbf", probability=True, class_weight=cw, random_state=SEED))]),
        {"pca__n_components": [10,20,30,40],
         "clf__C": [0.1,1,10,100], "clf__gamma": ["scale",0.01,0.1]},
    )

    # ── Logistic L1 ───────────────────────────────────────────────────────────
    spaces["logreg_l1"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                   *smote_steps,
                   ("clf", LogisticRegression(penalty="l1", solver="liblinear",
                            class_weight=cw, random_state=SEED, max_iter=5000))]),
        {"clf__C": [0.01,0.1,1,10]},
    )

    # ── Random Forest ─────────────────────────────────────────────────────────
    spaces["random_forest"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), *smote_steps,
                   ("clf", RandomForestClassifier(class_weight=cw, random_state=SEED, n_jobs=-1))]),
        {"clf__n_estimators": [200,400], "clf__max_depth": [None,5,10],
         "clf__min_samples_split": [2,5]},
    )

    # ── Extra Trees ───────────────────────────────────────────────────────────
    spaces["extra_trees"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), *smote_steps,
                   ("clf", ExtraTreesClassifier(class_weight=cw, random_state=SEED, n_jobs=-1))]),
        {"clf__n_estimators": [200,400], "clf__max_depth": [None,5,10],
         "clf__min_samples_split": [2,5]},
    )

    # ── Gradient Boosting ─────────────────────────────────────────────────────
    spaces["grad_boost"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                   *smote_steps,
                   ("clf", GradientBoostingClassifier(random_state=SEED))]),
        {"clf__n_estimators": [100,200], "clf__max_depth": [3,5],
         "clf__learning_rate": [0.05,0.1], "clf__subsample": [0.8,1.0]},
    )

    # ── MLP ───────────────────────────────────────────────────────────────────
    spaces["mlp"] = (
        PipeClass([("imp", SimpleImputer(strategy="median")),
                   ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                   ("pca", PCA(random_state=SEED)), *smote_steps,
                   ("clf", MLPClassifier(max_iter=500, random_state=SEED,
                            early_stopping=True, validation_fraction=0.15))]),
        {"pca__n_components": [20,30],
         "clf__hidden_layer_sizes": [(64,32),(128,64)],
         "clf__alpha": [1e-4,1e-3]},
    )

    # ── XGBoost ───────────────────────────────────────────────────────────────
    if HAS_XGB:
        spaces["xgboost"] = (
            PipeClass([("imp", SimpleImputer(strategy="median")),
                       ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                       ("pca", PCA(random_state=SEED)), *smote_steps,
                       ("clf", XGBClassifier(eval_metric="logloss",
                                scale_pos_weight=spw if is_imbalanced else 1.0,
                                random_state=SEED, n_jobs=-1))]),
            {"pca__n_components": [20,30,40],
             "clf__n_estimators": [100,200],
             "clf__max_depth": [3,5],
             "clf__learning_rate": [0.05,0.1],
             "clf__subsample": [0.8,1.0]},
        )

    # ── LightGBM ──────────────────────────────────────────────────────────────
    if HAS_LGB:
        spaces["lightgbm"] = (
            PipeClass([("imp", SimpleImputer(strategy="median")),
                       ("var", VarianceThreshold(1e-6)), ("scl", StandardScaler()),
                       ("pca", PCA(random_state=SEED)), *smote_steps,
                       ("clf", LGBMClassifier(class_weight=cw if not is_imbalanced else None,
                                is_unbalance=is_imbalanced,
                                random_state=SEED, n_jobs=-1, verbose=-1))]),
            {"pca__n_components": [20,30,40],
             "clf__n_estimators": [100,200],
             "clf__max_depth": [3,5],
             "clf__learning_rate": [0.05,0.1],
             "clf__num_leaves": [15,31]},
        )

    return spaces


In [9]:
# Stacking removed for lighter runtime


In [10]:
# ─── NESTED CV EXPERIMENT ─────────────────────────────────────────────────────
def nested_cv(X: np.ndarray, y: np.ndarray,
              task: TaskSpec, feature_set: str) -> pd.DataFrame:
    """Nested stratified CV with GridSearchCV inner loop."""
    outer_cv = StratifiedKFold(N_SPLITS_OUTER, shuffle=True, random_state=SEED)
    inner_cv = StratifiedKFold(N_SPLITS_INNER, shuffle=True, random_state=SEED)
    pos_ratio = float(y.mean())
    model_spaces = get_model_spaces(task.is_imbalanced, pos_ratio)
    rows = []

    for mname, (pipe, pgrid) in model_spaces.items():
        y_true_all, y_score_all = [], []
        skipped = 0
        for tr_idx, te_idx in outer_cv.split(X, y):
            Xtr, Xte = X[tr_idx], X[te_idx]
            ytr, yte = y[tr_idx], y[te_idx]
            ag = adapt_grid(mname, pgrid, Xtr)
            if ag is None: skipped += 1; continue
            try:
                gs = GridSearchCV(
                    clone(pipe), ag,
                    scoring="balanced_accuracy",
                    cv=inner_cv, n_jobs=-1,
                    refit=True, error_score=0.0)
                gs.fit(Xtr, ytr)
            except Exception:
                skipped += 1; continue
            try:
                y_score = gs.predict_proba(Xte)[:,1] if hasattr(gs,"predict_proba") \
                          else gs.decision_function(Xte)
                y_true_all.extend(yte); y_score_all.extend(y_score)
            except Exception:
                skipped += 1; continue

        if not y_true_all: continue
        yt = np.array(y_true_all); ys = np.array(y_score_all)
        thr = find_best_threshold(yt, ys) if (task.is_imbalanced and len(np.unique(yt))>1) else 0.5
        yp = (ys >= thr).astype(int)
        m = evaluate_metrics(yt, yp, ys)
        rows.append({"task": task.name, "feature_set": feature_set,
                     "model": mname, "threshold": round(thr,3),
                     "skipped_folds": skipped, **m})

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("accuracy", ascending=False)
    return df


In [11]:

# ─── GNN COMPONENTS ───────────────────────────────────────────────────────────
if HAS_PYG and HAS_TORCH:
    class GCNClassifier(nn.Module):
        def __init__(self, in_ch, hidden=GNN_HIDDEN, drop=0.3):
            super().__init__()
            self.conv1=GCNConv(in_ch,hidden); self.bn1=BatchNorm(hidden)
            self.conv2=GCNConv(hidden,hidden); self.bn2=BatchNorm(hidden)
            self.lin1=nn.Linear(hidden,hidden//2); self.lin2=nn.Linear(hidden//2,2)
            self.drop=drop
        def forward(self,x,edge_index,batch,ew=None):
            x=F.dropout(F.relu(self.bn1(self.conv1(x,edge_index,edge_weight=ew))),self.drop,self.training)
            x=F.relu(self.bn2(self.conv2(x,edge_index,edge_weight=ew)))
            x=global_mean_pool(x,batch)
            x=F.dropout(F.relu(self.lin1(x)),self.drop,self.training)
            return self.lin2(x)

    class GINClassifier(nn.Module):
        def __init__(self, in_ch, hidden=GNN_HIDDEN, drop=0.3):
            super().__init__()
            mlp1=nn.Sequential(nn.Linear(in_ch,hidden),nn.ReLU(),nn.Linear(hidden,hidden))
            mlp2=nn.Sequential(nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,hidden))
            self.conv1=GINConv(mlp1); self.bn1=BatchNorm(hidden)
            self.conv2=GINConv(mlp2); self.bn2=BatchNorm(hidden)
            self.lin1=nn.Linear(hidden,hidden//2); self.lin2=nn.Linear(hidden//2,2)
            self.drop=drop
        def forward(self,x,edge_index,batch,ew=None):
            x=F.dropout(F.relu(self.bn1(self.conv1(x,edge_index))),self.drop,self.training)
            x=F.relu(self.bn2(self.conv2(x,edge_index)))
            x=global_mean_pool(x,batch)
            x=F.dropout(F.relu(self.lin1(x)),self.drop,self.training)
            return self.lin2(x)

    def matrix_to_pyg(W: np.ndarray, label: int) -> "Data":
        rows,cols=np.where(W>0)
        edge_index=torch.tensor(np.stack([rows,cols]),dtype=torch.long)
        edge_attr=torch.tensor(W[rows,cols],dtype=torch.float).unsqueeze(1)
        x=torch.tensor(W,dtype=torch.float)
        return Data(x=x,edge_index=edge_index,edge_attr=edge_attr,
                    y=torch.tensor([label],dtype=torch.long))


In [12]:

# ─── GNN TRAINING & EVALUATION ────────────────────────────────────────────────
if HAS_PYG and HAS_TORCH:
    def _train_epoch(model,loader,opt,crit,device):
        model.train(); total=0.0
        for batch in loader:
            batch=batch.to(device); opt.zero_grad()
            loss=crit(model(batch.x,batch.edge_index,batch.batch,
                            getattr(batch,"edge_attr",None)),batch.y)
            loss.backward(); opt.step(); total+=loss.item()*batch.num_graphs
        return total/max(len(loader.dataset),1)

    @torch.no_grad()
    def _eval_loader(model,loader,device):
        model.eval(); yt,yp,ys=[],[],[]
        for batch in loader:
            batch=batch.to(device)
            logits=model(batch.x,batch.edge_index,batch.batch,
                         getattr(batch,"edge_attr",None))
            probs=F.softmax(logits,dim=1)[:,1]
            yt.extend(batch.y.cpu().numpy())
            yp.extend((probs>=0.5).long().cpu().numpy())
            ys.extend(probs.cpu().numpy())
        return np.array(yt),np.array(yp),np.array(ys)

    def _fit_gnn(model_cls,train_g,val_g,in_ch,pos_weight):
        device=DEVICE
        model=model_cls(in_ch).to(device)
        tr_l=PyGDataLoader(train_g,GNN_BATCH_SIZE,shuffle=True)
        va_l=PyGDataLoader(val_g,GNN_BATCH_SIZE,shuffle=False)
        w=torch.tensor([1.0,float(pos_weight)]).to(device)
        crit=nn.CrossEntropyLoss(weight=w)
        opt=torch.optim.Adam(model.parameters(),lr=GNN_LR,weight_decay=GNN_WEIGHT_DECAY)
        best_state,best_score,patience=None,-np.inf,0
        for ep in range(1,GNN_EPOCHS+1):
            _train_epoch(model,tr_l,opt,crit,device)
            yt,yp,_=_eval_loader(model,va_l,device)
            score=balanced_accuracy_score(yt,yp) if len(np.unique(yt))>1 else 0.0
            if score>best_score:
                best_score=score
                best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience=0
            else:
                patience+=1
            if patience>=GNN_PATIENCE: break
        if best_state: model.load_state_dict(best_state)
        return model

    def run_gnn(Ws,y,task):
        graphs=[matrix_to_pyg(W,int(lbl)) for W,lbl in zip(Ws,y)]
        in_ch=graphs[0].x.shape[1]
        pos_w=(len(y)-y.sum())/max(y.sum(),1)
        outer_cv=StratifiedKFold(N_SPLITS_OUTER, shuffle=True, random_state=SEED)
        rows=[]
        for ModelCls,mname in [(GCNClassifier,"gcn"),(GINClassifier,"gin")]:
            yt_all,yp_all,ys_all=[],[],[]
            for fold,(tr_idx,te_idx) in enumerate(outer_cv.split(np.zeros(len(y)),y)):
                tr_idx=np.array(tr_idx)
                inner=StratifiedKFold(4,shuffle=True,random_state=SEED+fold)
                sub_tr,sub_val=next(inner.split(np.zeros(len(tr_idx)),y[tr_idx]))
                tr2,val2=tr_idx[sub_tr],tr_idx[sub_val]
                model=_fit_gnn(ModelCls,[graphs[i] for i in tr2],
                               [graphs[i] for i in val2],in_ch,pos_w)
                te_l=PyGDataLoader([graphs[i] for i in te_idx],GNN_BATCH_SIZE)
                yt,yp,ys=_eval_loader(model,te_l,DEVICE)
                yt_all.extend(yt); yp_all.extend(yp); ys_all.extend(ys)
                del model; gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            yt_all=np.array(yt_all); ys_all=np.array(ys_all)
            if task.is_imbalanced and len(np.unique(yt_all))>1:
                thr=find_best_threshold(yt_all,ys_all)
                yp_all=(ys_all>=thr).astype(int)
            else:
                thr=0.5; yp_all=np.array(yp_all)
            m=evaluate_metrics(yt_all,yp_all,ys_all)
            rows.append({"task":task.name,"feature_set":"graph_input",
                         "model":mname,"threshold":round(thr,3),**m})
        return pd.DataFrame(rows)


In [13]:

# ─── RESULTS PRINTING ─────────────────────────────────────────────────────────
def print_results(df: pd.DataFrame, feature_set: str):
    sub = df[df["feature_set"]==feature_set].copy() if "feature_set" in df.columns else df.copy()
    if sub.empty: return
    sub = sub.sort_values("accuracy", ascending=False)
    print(f"\n  [{feature_set}]")
    print(f"  {'Model':<22} {'Acc':>8} {'BalAcc':>10} {'F1':>8}")
    print("  " + "-"*52)
    for _, r in sub.iterrows():
        print(f"  {r['model']:<22} {r['accuracy']:>8.3f} {r['balanced_accuracy']:>10.3f} {r['f1']:>8.3f}")

def print_task_summary(all_dfs: List[pd.DataFrame], task_name: str):
    df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n{'='*75}")
    print(f"  TASK: {task_name}")
    print(f"{'='*75}")
    for fs in df["feature_set"].unique():
        print_results(df, fs)
    best = df.loc[df["accuracy"].idxmax()]
    print(f"\n  ★ BEST  model={best['model']},  feature_set={best['feature_set']},  "
          f"Acc={best['accuracy']:.3f},  BalAcc={best['balanced_accuracy']:.3f},  F1={best['f1']:.3f}")


In [14]:

# ─── VISUAL ANALYTICS ─────────────────────────────────────────────────────────
def visualize_networks(Ws: np.ndarray, y: np.ndarray, task_name: str, class_names: List[str]):
    idx = np.triu_indices(Ws.shape[1], k=1)
    X0e = Ws[y==0][:,idx[0],idx[1]]
    X1e = Ws[y==1][:,idx[0],idx[1]]
    tvals, pvals = ttest_ind(X1e, X0e, axis=0, equal_var=False, nan_policy="omit")
    _, qvals = fdrcorrection(np.nan_to_num(pvals, nan=1.0))
    sig = np.sum(qvals < 0.05)
    print(f"  Significant edges (FDR q<0.05): {sig}/{len(pvals)}")


In [15]:

# ─── MAIN PIPELINE ────────────────────────────────────────────────────────────
def run_task(bundle: DatasetBundle, task: TaskSpec):
    Ws      = bundle.W_prob[task.mask]
    X_edges = bundle.X_edges[task.mask]
    X_graph = bundle.X_graph.loc[task.mask].values
    X_spec  = bundle.X_spectral[task.mask]
    X_asym  = bundle.X_asym[task.mask]
    X_all   = bundle.X_all[task.mask]
    X_topo  = np.hstack([X_graph, X_spec, X_asym])  # topology only (no raw edges)
    y = task.y

    print(f"\n\n{'#'*75}")
    print(f"#  Running: {task.name}   (n={len(y)}, minority={y.sum()})")
    print(f"{'#'*75}")
    visualize_networks(Ws, y, task.name, task.class_names)

    all_dfs = []

    print("\n--- Edge Features ---")
    all_dfs.append(nested_cv(X_edges, y, task, "edge_upper_triangle"))

    print("--- Topology Features (graph+spectral+asym) ---")
    all_dfs.append(nested_cv(X_topo, y, task, "topology"))

    print("--- All Features Combined ---")
    all_dfs.append(nested_cv(X_all, y, task, "all_combined"))

    if HAS_PYG and HAS_TORCH:
        print("--- GNN (GCN + GIN) ---")
        df_gnn = run_gnn(Ws, y, task)
        if not df_gnn.empty: all_dfs.append(df_gnn)

    print_task_summary(all_dfs, task.name)
    return pd.concat(all_dfs, ignore_index=True)

def main():
    print("Loading dataset...")
    bundle = load_dataset(MAT_DIR, META_PATH)
    print("\nBuilding tasks...")
    tasks = build_tasks(bundle.df)
    all_results = []
    for task in tasks:
        res = run_task(bundle, task)
        all_results.append(res)
    global grand
    grand = pd.concat(all_results, ignore_index=True)
    print(f"\n\n{'='*75}")
    print("  GRAND SUMMARY — Top model per task")
    print(f"{'='*75}")
    for task_name, grp in grand.groupby("task"):
        best = grp.loc[grp["accuracy"].idxmax()]
        print(f"  {task_name:<40}  Acc={best['accuracy']:.3f}  "
              f"BalAcc={best['balanced_accuracy']:.3f}  F1={best['f1']:.3f}  "
              f"model={best['model']}  feat={best['feature_set']}")
    print("\n[Done]")

main()


Loading dataset...
Loaded 114 subjects
  Edge features:      2415
  Graph+Spec stats:   63
  Asymmetry:          10
  TOTAL combined:     2488

Building tasks...
  sex_female_vs_male: n=114, class0=50, class1=64
  creativity_normal_vs_creative: n=94, class0=67, class1=27
  math_normal_vs_high: n=87, class0=67, class1=20


###########################################################################
#  Running: sex_female_vs_male   (n=114, minority=64)
###########################################################################
  Significant edges (FDR q<0.05): 0/2415

--- Edge Features ---
--- Topology Features (graph+spectral+asym) ---
--- All Features Combined ---
--- GNN (GCN + GIN) ---

  TASK: sex_female_vs_male

  [edge_upper_triangle]
  Model                       Acc     BalAcc       F1
  ----------------------------------------------------
  grad_boost                0.693      0.676    0.748
  random_forest             0.667      0.646    0.732
  logreg_l1                 0.658 

c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: U

--- All Features Combined ---


c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty=

--- GNN (GCN + GIN) ---


c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\vishn\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch_g


  TASK: math_normal_vs_high

  [edge_upper_triangle]
  Model                       Acc     BalAcc       F1
  ----------------------------------------------------
  extra_trees               0.793      0.550    0.182
  xgboost                   0.770      0.588    0.333
  logreg_l1                 0.747      0.503    0.083
  lightgbm                  0.644      0.576    0.367
  mlp                       0.575      0.584    0.393
  random_forest             0.575      0.584    0.393
  svm_mi                    0.529      0.571    0.388
  grad_boost                0.517      0.581    0.400
  svm_pca                   0.471      0.587    0.410

  [topology]
  Model                       Acc     BalAcc       F1
  ----------------------------------------------------
  grad_boost                0.759      0.545    0.222
  xgboost                   0.494      0.601    0.421
  extra_trees               0.483      0.612    0.430
  mlp                       0.460      0.579    0.405
  svm_pca   

In [16]:

# ─── BEST SCORES PER TASK ─────────────────────────────────────────────────────
def print_best_scores(grand: pd.DataFrame):
    print(f"\n{'='*75}")
    print(f"  BEST SCORES PER TASK")
    print(f"{'='*75}")
    print(f"  {'Task':<35} {'Acc':>8} {'BalAcc':>8} {'F1':>8}  Model / Features")
    print("  " + "-"*85)
    for task_name, grp in grand.groupby("task"):
        best = grp.loc[grp["accuracy"].idxmax()]
        print(f"  {task_name:<35} {best['accuracy']:>8.3f} {best['balanced_accuracy']:>8.3f} "
              f"{best['f1']:>8.3f}  {best['model']} / {best['feature_set']}")
    print()

print_best_scores(grand)



  BEST SCORES PER TASK
  Task                                     Acc   BalAcc       F1  Model / Features
  -------------------------------------------------------------------------------------
  creativity_normal_vs_creative          0.723    0.519    0.071  grad_boost / topology
  math_normal_vs_high                    0.793    0.550    0.182  extra_trees / edge_upper_triangle
  sex_female_vs_male                     0.693    0.676    0.748  grad_boost / edge_upper_triangle

